In [31]:
import os
import json
import pandas as pd
import numpy as np
from tqdm import tqdm 

In [2]:
icd9 = pd.read_json('../data/icd9-cm-2015-code.json').T
icd9 = icd9.reset_index(names="Code")
title = icd9[["Code", "title", "synonym"]].copy()
title = title.replace("Other", np.nan)
title['title'] = title['title'].fillna(title['synonym'])
title['title'] = title['title'].apply(lambda x : x[0] if isinstance(x, list) else x)
title = title.drop(columns=['synonym'])
title

/var/folders/h_/95dmw6rj241f_n541xn2p4xw0000gn/T/ipykernel_47492/2219855615.py:1: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  icd9 = pd.read_json('../data/icd9-cm-2015-code.json').T
/var/folders/h_/95dmw6rj241f_n541xn2p4xw0000gn/T/ipykernel_47492/2219855615.py:1: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  icd9 = pd.read_json('../data/icd9-cm-2015-code.json').T
/var/folders/h_/95dmw6rj241f_n541xn2p4xw0000gn/T/ipykernel_47492/2219855615.py:1: FutureWarning: The behavior of 'to_d

,Code,title
0,00,"Procedures and interventions , Not Elsewhere C..."
1,000,Therapeutic ultrasound
2,0001,Therapeutic ultrasound of vessels of head and ...
3,0002,Therapeutic ultrasound of heart
4,0003,Therapeutic ultrasound of peripheral vascular ...
...,...,...
4641,9995,Stretching of foreskin
4642,9996,Collection of sperm for artificial insemination
4643,9997,Fitting of denture
4644,9998,Extraction of milk from lactating breast


In [3]:
methods = ["LavenDist", "PubMedBERT", "TFIDF"]
df = {m : pd.read_excel(f"../result/icd9_inclusion_mapping_summary_{m}.xlsx", dtype='string') 
      for m in methods}
df = pd.concat(df, names=["method"]).reset_index(level="method").reset_index(drop=True)
df['Code_include'] = df['Code_include'].str.split('. ')
df = df.explode('Code_include')
df

,method,Code,title,include,Code_include
0,LavenDist,07,Operations on other endocrine glands,operations on: pineal gland,0759
1,LavenDist,07,Operations on other endocrine glands,operations on: pituitary gland,0779
2,LavenDist,07,Operations on other endocrine glands,operations on: thymus,0141
2,LavenDist,07,Operations on other endocrine glands,operations on: thymus,0799
3,LavenDist,143,Repair of retinal tear,repair of retinal defect,3570
...,...,...,...,...,...
136,TFIDF,887,Diagnostic ultrasound,Non-invasive ultrasound,8872
137,TFIDF,887,Diagnostic ultrasound,Ultrasonography,8872
137,TFIDF,887,Diagnostic ultrasound,Ultrasonography,8873
137,TFIDF,887,Diagnostic ultrasound,Ultrasonography,9513


In [64]:
df_all_method = df.groupby(by=['Code', 'title', 'include'],as_index=False).agg(set)
df_all_method = df_all_method.explode('Code_include')

# Exclude code include to self
mask = df_all_method.apply(lambda row: str(row['Code_include']).startswith(str(row['Code'])), axis=1)
df_all_method = df_all_method[~mask]
df_all_method

,Code,title,include,method,Code_include
2,07,Operations on other endocrine glands,operations on: thymus,"{LavenDist, TFIDF, PubMedBERT}",0141
2,07,Operations on other endocrine glands,operations on: thymus,"{LavenDist, TFIDF, PubMedBERT}",11
2,07,Operations on other endocrine glands,operations on: thymus,"{LavenDist, TFIDF, PubMedBERT}",147
2,07,Operations on other endocrine glands,operations on: thymus,"{LavenDist, TFIDF, PubMedBERT}",56
4,143,Repair of retinal tear,repair of retinal defect,{LavenDist},3570
...,...,...,...,...,...
75,886,Phlebography,venography using contrast material,{PubMedBERT},885
75,886,Phlebography,venography using contrast material,{PubMedBERT},884
77,887,Diagnostic ultrasound,Ultrasonography,"{LavenDist, TFIDF}",9513
78,967,Other continuous invasive mechanical ventilation,BiPAP delivered through endotracheal tube or t...,{PubMedBERT},9390


In [46]:
df_all_method.columns

Index(['Code', 'title', 'include', 'method', 'Code_include'], dtype='object')

In [65]:
df_prep_ai = df_all_method.drop(columns=['method'])
df_prep_ai = df_prep_ai.drop_duplicates()
df_prep_ai = df_prep_ai.merge(title, how='left', left_on='Code_include', right_on='Code', suffixes=("","_Code_include"))
df_prep_ai = df_prep_ai.drop(columns=['Code_Code_include'])
df_prep_ai

,Code,title,include,Code_include,title_Code_include
0,07,Operations on other endocrine glands,operations on: thymus,0141,Operations on thalamus
1,07,Operations on other endocrine glands,operations on: thymus,11,Operations on cornea
2,07,Operations on other endocrine glands,operations on: thymus,147,Operations on vitreous
3,07,Operations on other endocrine glands,operations on: thymus,56,Operations on ureter
4,143,Repair of retinal tear,repair of retinal defect,3570,Other and unspecified repair of unspecified se...
...,...,...,...,...,...
130,886,Phlebography,venography using contrast material,885,Angiocardiography using contrast material
131,886,Phlebography,venography using contrast material,884,Arteriography using contrast material
132,887,Diagnostic ultrasound,Ultrasonography,9513,Ultrasound study of eye
133,967,Other continuous invasive mechanical ventilation,BiPAP delivered through endotracheal tube or t...,9390,Non-invasive mechanical ventilation


In [ ]:
all_input = []
for index, row in tqdm(df_prep_ai.iterrows(), total=len(df_prep_ai)):
    
    p = f"""Parent Category:
- Title: {row['title']}
- Inclusion terms: {row['include']}

Target Code to verify:
- Title: {row['title_Code_include']}"""
    all_input.append(p)

Processing 135 rows...


100%|██████████| 135/135 [00:00<00:00, 7972.17it/s]


In [29]:
for i in range(0, len(all_input), 10):
    chunk = all_input[i:i+10]
    file_num = (i // 10) + 1  
    file_path = f"../result/text_prompt/{file_num}.txt"
    with open(file_path, "w", encoding="utf-8") as file:
        file.write('\n===\n'.join(chunk))

In [37]:
# Load response
# https://share.gemini.google/NrDO1HABTm6X

response_path = '../result/response/'
response  = []
for f in os.listdir(response_path):
    path = os.path.join(response_path, f)
    with open(path, 'r', encoding='utf-8') as file:
        data = json.load(file)
        response.extend(data)

In [66]:
df_response = pd.DataFrame(response)
df_ai = pd.concat([df_prep_ai, df_response], axis=1)
df_ai

,Code,title,include,Code_include,title_Code_include,reasoning,is_correct
0,07,Operations on other endocrine glands,operations on: thymus,0141,Operations on thalamus,Tricuspid valve replacement is distinct from a...,False
1,07,Operations on other endocrine glands,operations on: thymus,11,Operations on cornea,Unspecified heart valve is outside the specifi...,False
2,07,Operations on other endocrine glands,operations on: thymus,147,Operations on vitreous,Pulmonary valve replacement is distinct from a...,False
3,07,Operations on other endocrine glands,operations on: thymus,56,Operations on ureter,Mitral valve replacement is distinct from aort...,False
4,143,Repair of retinal tear,repair of retinal defect,3570,Other and unspecified repair of unspecified se...,Tricuspid valve replacement is distinct from a...,False
...,...,...,...,...,...,...,...
130,886,Phlebography,venography using contrast material,885,Angiocardiography using contrast material,Mitral valve replacement is distinct from tric...,False
131,886,Phlebography,venography using contrast material,884,Arteriography using contrast material,Unspecified heart valve is outside the specifi...,False
132,887,Diagnostic ultrasound,Ultrasonography,9513,Ultrasound study of eye,Pulmonary valve replacement is distinct from t...,False
133,967,Other continuous invasive mechanical ventilation,BiPAP delivered through endotracheal tube or t...,9390,Non-invasive mechanical ventilation,Aortic valve replacement is distinct from tric...,False


In [57]:
df_ai.to_excel('../result/icd9_inclusion_mapping_summary.xlsx', index = False)

In [59]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# 1. Load your existing Excel file
file_path = '../result/icd9_inclusion_mapping_summary.xlsx'
wb = load_workbook(file_path)

print(f"Applying formatting to {file_path}...")

# 2. Loop through all the sheets in your workbook
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    
    last_col_idx = ws.max_column

    # --- BEATIFICATION STEP A: Auto-fit Column Widths ---
    for col in ws.columns:
        max_length = 0
        column = col[0].column_letter # Gets the letter (A, B, C...)
        
        # Find the longest string of text in the entire column
        for cell in col:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        
        # Add a little padding (+2) and set the new width. 
        # (We cap it at 75 so extremely long inclusion texts don't stretch the screen too far)
        adjusted_width = min(max_length + 2, 75) 
        ws.column_dimensions[column].width = adjusted_width

    # --- BEATIFICATION STEP B: Style the Header Row ---
    # Create a light gray fill and a bold font
    header_fill = PatternFill(start_color="E0E0E0", end_color="E0E0E0", fill_type="solid")
    header_font = Font(bold=True, color="000000")

    # Apply the styling to every cell in the first row
    for cell in ws[1]: 
        cell.font = header_font
        cell.fill = header_fill
        # Center the text and wrap it if it gets too long
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        
    # --- BEATIFICATION STEP C: Highlight last column if True ---
    # Loop down rows starting from row 2 (skipping header)
    for row in range(2, ws.max_row + 1):
        cell = ws.cell(row=row, column=last_col_idx)
        
        # Check if value is boolean True or text string 'True' / 'TRUE'
        if cell.value is True or str(cell.value).strip().upper() == 'TRUE':
            cell.fill = PatternFill(start_color="E2F0D9", end_color="E2F0D9", fill_type="solid")

# 3. Save the changes back to the file
wb.save(file_path)
print("Excel file successfully beautified!")

Applying formatting to ../result/icd9_inclusion_mapping_summary.xlsx...
Excel file successfully beautified!


In [67]:
df_ai_method = df_ai.merge(df_all_method, how='inner', on=['Code', 'title', 'include', 'Code_include'])
df_ai_method

,Code,title,include,Code_include,title_Code_include,reasoning,is_correct,method
0,07,Operations on other endocrine glands,operations on: thymus,0141,Operations on thalamus,Tricuspid valve replacement is distinct from a...,False,"{LavenDist, TFIDF, PubMedBERT}"
1,07,Operations on other endocrine glands,operations on: thymus,11,Operations on cornea,Unspecified heart valve is outside the specifi...,False,"{LavenDist, TFIDF, PubMedBERT}"
2,07,Operations on other endocrine glands,operations on: thymus,147,Operations on vitreous,Pulmonary valve replacement is distinct from a...,False,"{LavenDist, TFIDF, PubMedBERT}"
3,07,Operations on other endocrine glands,operations on: thymus,56,Operations on ureter,Mitral valve replacement is distinct from aort...,False,"{LavenDist, TFIDF, PubMedBERT}"
4,143,Repair of retinal tear,repair of retinal defect,3570,Other and unspecified repair of unspecified se...,Tricuspid valve replacement is distinct from a...,False,{LavenDist}
...,...,...,...,...,...,...,...,...
130,886,Phlebography,venography using contrast material,885,Angiocardiography using contrast material,Mitral valve replacement is distinct from tric...,False,{PubMedBERT}
131,886,Phlebography,venography using contrast material,884,Arteriography using contrast material,Unspecified heart valve is outside the specifi...,False,{PubMedBERT}
132,887,Diagnostic ultrasound,Ultrasonography,9513,Ultrasound study of eye,Pulmonary valve replacement is distinct from t...,False,"{LavenDist, TFIDF}"
133,967,Other continuous invasive mechanical ventilation,BiPAP delivered through endotracheal tube or t...,9390,Non-invasive mechanical ventilation,Aortic valve replacement is distinct from tric...,False,{PubMedBERT}


In [69]:
overall_total = len(df_ai_method)
overall_correct = df_ai_method['is_correct'].sum()
overall_accuracy = (overall_correct / overall_total) * 100 if overall_total > 0 else 0

print("=========================================")
print(f"OVERALL PERFORMANCE")
print("=========================================")
print(f"Total Rows Evaluated: {overall_total}")
print(f"Total Correct Maps : {overall_correct}")
print(f"Overall Accuracy   : {overall_accuracy:.2f}%\n")


# 2. Calculate Accuracy Per Method
# We explode the 'method' column because a single row can belong to multiple methods
df_exploded = df_ai_method.explode('method')

# Ensure boolean evaluation is ready
df_exploded['is_correct'] = df_exploded['is_correct'].astype(bool)

# Group by the individual methods to aggregate counts and accuracy
method_evaluation = df_exploded.groupby('method').agg(
    Total_Predictions=('is_correct', 'count'),
    Correct_Predictions=('is_correct', 'sum')
).reset_index()

# Compute the accuracy percentage for each method
method_evaluation['Accuracy (%)'] = (
    method_evaluation['Correct_Predictions'] / method_evaluation['Total_Predictions'] * 100
).round(2)

# Sort by highest accuracy first
method_evaluation = method_evaluation.sort_values(by='Accuracy (%)', ascending=False)

print("=========================================")
print("PERFORMANCE PER METHOD")
print("=========================================")
print(method_evaluation.to_string(index=False))

OVERALL PERFORMANCE
Total Rows Evaluated: 135
Total Correct Maps : 33
Overall Accuracy   : 24.44%

PERFORMANCE PER METHOD
    method  Total_Predictions  Correct_Predictions  Accuracy (%)
PubMedBERT                124                   33         26.61
 LavenDist                 64                    9         14.06
     TFIDF                 51                    3          5.88
